In [11]:
import sys, os
from tqdm import tqdm
import time

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.append(PROJECT_ROOT)

import torch.nn as nn
import torch
from torch.utils.data import DataLoader
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from data import MessageDataset, setup_tokenizer, get_json_files, split_dataset, collate_fn, create_input_target_pairs

In [21]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
setup_tokenizer(tokenizer)

json_files = get_json_files(os.path.join(PROJECT_ROOT, "data"))
full_dataset = MessageDataset(json_files, tokenizer=tokenizer, sequence_length=256)
train_dataset, val_dataset, test_dataset = split_dataset(full_dataset)

dataloader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=lambda batch: collate_fn(batch, tokenizer.pad_token_id),
)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print(f"Vocab size: {len(tokenizer)}")
print(f"Dataset: {len(full_dataset)} total | {len(train_dataset)} train | {len(val_dataset)} val | {len(test_dataset)} test")

Vocab size: 50266
Dataset: 16790 total | 13432 train | 1679 val | 1679 test


In [22]:
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"GPT-2 Small: {total_params/1e6:.1f}M params")

GPT-2 Small: 124.4M params


In [23]:
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50266, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50266, bias=False)
)


In [24]:
for name, param in model.named_parameters():
    print(f"{name:60s} {str(list(param.shape)):20s} {param.numel():>10,}  {'trainable' if param.requires_grad else 'frozen'}")


transformer.wte.weight                                       [50266, 768]         38,604,288  trainable
transformer.wpe.weight                                       [1024, 768]             786,432  trainable
transformer.h.0.ln_1.weight                                  [768]                       768  trainable
transformer.h.0.ln_1.bias                                    [768]                       768  trainable
transformer.h.0.attn.c_attn.weight                           [768, 2304]           1,769,472  trainable
transformer.h.0.attn.c_attn.bias                             [2304]                    2,304  trainable
transformer.h.0.attn.c_proj.weight                           [768, 768]              589,824  trainable
transformer.h.0.attn.c_proj.bias                             [768]                       768  trainable
transformer.h.0.ln_2.weight                                  [768]                       768  trainable
transformer.h.0.ln_2.bias                                    [76

In [25]:
import torch.nn as nn

for param in model.parameters():
    param.requires_grad = False

class LoRALayer(nn.Module):
    def __init__(self, original_layer, r=2):
        super().__init__()
        self.original_layer = original_layer

        in_shape = original_layer.weight.shape[0] 
        out_shape = original_layer.weight.shape[1]

        self.A = nn.Parameter(torch.empty(in_shape, r))
        nn.init.kaiming_uniform_(self.A)
        self.B = nn.Parameter(torch.zeros(r, out_shape))

    def forward(self, x):
        return self.original_layer(x) + x @ self.A @ self.B



In [26]:
# Replace model layers with LoRA layers
for block in model.transformer.h:
    block.attn.c_attn = LoRALayer(block.attn.c_attn)
    block.mlp.c_fc = LoRALayer(block.mlp.c_fc)
    block.mlp.c_proj = LoRALayer(block.mlp.c_proj)

model = model.to(DEVICE)


In [27]:
# Training Definition
num_epochs = 3
lr = 3e-4

# still exclude layers from adam to save memory
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs * len(dataloader))
loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

In [28]:

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=lambda batch: collate_fn(batch, tokenizer.pad_token_id),
)

start = time.time()

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for batch in tqdm(dataloader, desc=f"Epoch {epoch} [train]"):
         optimizer.zero_grad()
         x, y = create_input_target_pairs(batch.to(DEVICE))
         pred = model(x).logits
         loss = loss_fn(pred.reshape(-1, pred.size(-1)), y.reshape(-1))
         epoch_loss += loss.item()
         loss.backward()
         optimizer.step()
         scheduler.step()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            x, y = create_input_target_pairs(batch.to(DEVICE))
            pred = model(x).logits
            val_loss += loss_fn(pred.reshape(-1, pred.size(-1)), y.reshape(-1)).item()

    print(f"Epoch {epoch}: train={epoch_loss / len(dataloader):.4f} val={val_loss / len(val_loader):.4f} lr={scheduler.get_last_lr()[0]:.2e}")

print(f"Model training complete in {time.time() - start:.1f}s")

Epoch 0 [train]:   8%|▊         | 130/1679 [02:00<23:58,  1.08it/s]


KeyboardInterrupt: 